# HackODS UNAM 2026: Pipeline de Datos y Análisis Exploratorio (Equipo linuxitOS)

Este notebook contiene el pipeline ETL (Extract, Transform, Load) centralizado que alimenta el [Dashboard ODS 1 x ODS 6](https://github.com/linuxitOS-hackODS/linuxitOS-HackODS-UNAM/tree/main/dashboard).
    
El objetivo principal es demostrar la transparencia y reproducibilidad técnica de nuestro análisis de datos geoespaciales, sociodemográficos y financieros. Para facilitar la evaluación de nuestro repositorio sin perder la comprensión de nuestra arquitectura técnica, hemos consolidado la ejecución en un orquestador principal (`main.py`).

## Arquitectura del Pipeline ETL (7 Módulos)
Nuestro procesamiento de datos está dividido en 7 fases especializadas que se ejecutan de manera secuencial. A continuación se describe el propósito de cada script de la carpeta `scripts/`:

1. **`siods_extractor.py`**: Recolección de métricas de infraestructura municipal desde SIODS.
2. **`analyze_iter.py`**: Depuración demográfica y cálculo de ruralidad desde el ITER (INEGI).
3. **`clean_coneval.py`**: Normalización de Índices de Rezago Social y Pobreza (CONEVAL).
4. **`merge_inegi_coneval.py`**: Integración o *merge* de la base demográfica.
5. **`merge_shcp.py`**: Cruce financiero con el presupuesto federal (FAISMUN/FORTAMUN).
6. **`merge_conagua.py`**: Cruce hídrico de disponibilidad por regiones hidrológicas (CONAGUA).
7. **`simplify_geojson.py`**: Optimización algorítmica geoespacial (Douglas-Peucker) para reducir los polígonos de cuencas de 18MB a 1MB y garantizar alto rendimiento web.

## Ejecución del Orquestador
La celda de código a continuación corresponde a nuestro `main.py`, el cual invoca secuencialmente cada uno de estos módulos, construye el dataset analítico y lo exporta para consumo nativo en el framework Quarto.


In [1]:
import subprocess
import sys
import os

def run_script(script_path):
    print(f"\n{'-'*50}")
    print(f"[INFO] Ejecutando módulo: {script_path}")
    print(f"{'-'*50}")
    try:
        # Usar subprocess para llamar al script de python
        result = subprocess.run([sys.executable, script_path], check=True)
    except subprocess.CalledProcessError as e:
        print(f"\n[ERROR] Falla en la ejecución de {script_path}. Abortando pipeline.")
        sys.exit(1)
    except Exception as e:
        print(f"\n[FATAL] Error inesperado en el módulo {script_path}: {e}")
        sys.exit(1)

def main():
    print("Iniciando Pipeline Automatizado de Datos (ETL): HackODS UNAM 2026 - linuxitOS\n")
    
    # Ajuste dinámico del directorio de trabajo (soporte para Terminal vs Jupyter Notebooks)
    # Si Jupyter lanza la libreta desde la carpeta 'notebooks/', bajamos un nivel para poder ver 'scripts/'.
    if not os.path.exists('scripts') and os.path.exists(os.path.join('..', 'scripts')):
        os.chdir('..')
        print("[INFO] Entorno Jupyter Notebook detectado. El directorio de trabajo se ajustó a la raíz del proyecto.\n")
    elif not os.path.exists('scripts'):
         print("[ERROR] El entorno no pudo localizar la carpeta 'scripts/'.")
         print("        Asegúrate de ejecutar desde la raíz del repositorio o dentro de la carpeta 'notebooks/'.")
         sys.exit(1)

    # Orden secuencial de ejecución para la integración del dataset maestro
    pipeline = [
        "scripts/siods_extractor.py",
        "scripts/analyze_iter.py",
        "scripts/clean_coneval.py",
        "scripts/merge_inegi_coneval.py",
        "scripts/merge_shcp.py",
        "scripts/merge_conagua.py",
        "scripts/simplify_geojson.py"
    ]

    for script in pipeline:
        run_script(script)

    print("\n[ÉXITO] Pipeline ETL concluido satisfactoriamente.")
    print("[INFO] El dataset analítico 'final_merged_data.csv' ha sido generado exitosamente.")
    print("[INFO] Para compilar el dashboard Quarto, ejecute el siguiente comando:")
    print("       quarto render dashboard/index.qmd")

if __name__ == "__main__":
    main()

# Para ejecutar el pipeline completo desde esta libreta, 
# descomente la siguiente línea si el entorno es adecuado:
# main()


Iniciando Pipeline Automatizado de Datos (ETL): HackODS UNAM 2026 - linuxitOS

[INFO] Entorno Jupyter Notebook detectado. El directorio de trabajo se ajustó a la raíz del proyecto.


--------------------------------------------------
[INFO] Ejecutando módulo: scripts/siods_extractor.py
--------------------------------------------------


[INFO] Iniciando extracción de datos desde API SIODS (Agenda 2030).
[INFO] Procesando indicador: ODS_6.1.1.a_Nacional_Agua_Segura
[INFO] Procesando indicador: ODS_6.1.1.c_Estatal_Agua_Segura

[ÉXITO] Extracción finalizada. Dataset original guardado en: ../datos/siods_agua_saneamiento_nacional.csv

[INFO] --- Resumen Estadístico y Granularidad ---
[INFO] Entidades geográficas únicas registradas: 33
[INFO] Detalle: La matriz contiene inferencia a nivel Nacional y Estatal. Requisito pivotar hacia CONEVAL para resolver déficit de datos a nivel municipal.

--------------------------------------------------
[INFO] Ejecutando módulo: scripts/analyze_iter.py
--------------------------------------------------


[INFO] Cargando microdatos censales (ITER 2020)...

[INFO] --- Resumen Poblacional (Censo 2020) ---
[INFO] Municipios totales evaluados: 2469
[INFO] Entidades de vocación rural predominante: 1360
[INFO] Entidades de vocación urbana predominante: 1109
[INFO] Universo poblacional censado: 126,411,503
[INFO] Población clasificada como rural: 27,338,731
[INFO] Tasa global de ruralidad: 21.63%

--------------------------------------------------
[INFO] Ejecutando módulo: scripts/clean_coneval.py
--------------------------------------------------


[INFO] Leyendo matriz CONEVAL: datos/Concentrado_indicadores_de_pobreza_2020.xlsx
[ÉXITO] Matriz CONEVAL sanitizada y almacenada en: datos/coneval_clean_2020.csv

[INFO] --- Resumen Descriptivo (CONEVAL) ---
       Pobreza_pct  Pobreza_extrema_pct  Carencia_servicios_pct
count  2466.000000          2466.000000             2466.000000
mean     62.002065            17.236355               40.134817
std      21.903723            15.323598               29.865334
min       5.450951             0.000000                0.084550
25%      45.580691             5.354647               11.440258
50%      62.745101            12.537985               35.090204
75%      80.316135            24.275407               66.028980
max      99.646676            84.447850              100.000000

--------------------------------------------------
[INFO] Ejecutando módulo: scripts/merge_inegi_coneval.py
--------------------------------------------------


[INFO] 1. Agregando y calculando matriz de ruralidad municipal (Censo INEGI 2020)
[INFO] 2. Incorporando vector de pobreza multidimensional CONEVAL: ../datos/coneval_clean_2020.csv
[INFO] 3. Inicializando join relacional (Inner Merge) entre bases sociodemográficas...

[ÉXITO] Matriz transaccional generada en: ../datos/final_merged_data.csv
[INFO] Observaciones integradas: 2469

[INFO] --- Vista Preliminar (Head) ---
  CVEGEO       Municipio  Pobreza_pct  Pobreza_extrema_pct  Carencia_servicios_pct          Estado  poblacion_total  pct_rural clasificacion_rural
0  01001  Aguascalientes    23.682258             1.974081                1.135570  Aguascalientes           951537   5.264220              Urbano
1  01002        Asientos    40.131881             4.142806                7.411217  Aguascalientes            51896  65.777709               Rural
2  01003        Calvillo    45.755944             4.498973                3.078508  Aguascalientes            58486  51.497794             

[INFO] Instanciando dataframes (Dataset Maestro y SHCP)...
[INFO] Limpiando base de derechos de agua SHCP (Corte: 2020)...
[INFO] Ejecutando operación Left Join con estructura base...
[INFO] Escribiendo matriz transaccional en /home/sebs/Documentos/HackODS/linuxitOS-HackODS2026/datos/final_merged_data.csv...
[ÉXITO] Dimensiones financieras integradas al espacio matricial.
[INFO] Verificando cabeceras financieras (n=5):
  CVEGEO       Municipio   monto_agua  tomas_pagadas
0  01001  Aguascalientes  893532100.0       242484.0
1  01002        Asientos    2961854.0         2989.0
2  01003        Calvillo   38814809.0        16877.0
3  01004           Cosío    1047418.0          774.0
4  01005     Jesús María  103769889.0        28630.0

--------------------------------------------------
[INFO] Ejecutando módulo: scripts/merge_conagua.py
--------------------------------------------------


[INFO] Leyendo base: final_merged_data.csv
[INFO] Leyendo base: CONAGUA (Población con acceso al agua 2020)
[INFO] Concatenando observaciones (Left Join por CVEGEO)...
[INFO] Observaciones iniciales: 2469
[INFO] Observaciones resultantes: 2469
[INFO] Verificación taxonómica (head):
  CVEGEO       Municipio  ... cobertura_agua_conagua_pct  carencia_agua_conagua_pct
0  01001  Aguascalientes  ...                      99.51                       0.49
1  01002        Asientos  ...                      99.09                       0.91
2  01003        Calvillo  ...                      99.13                       0.87

[3 rows x 5 columns]
[INFO] Volcando estructura consolidada en disco (CSV)...
[ÉXITO] Base CONAGUA unificada con dataset maestro.

--------------------------------------------------
[INFO] Ejecutando módulo: scripts/simplify_geojson.py
--------------------------------------------------


[INFO] Cargando topología: Disponibilidad_Cuencas_2020.json
[INFO] Aplicando Douglas-Peucker: reduccion poligonal para Disponibilidad_Cuencas_2020.json
[INFO] Vaciando vector espacial optimizado a io stream: Disponibilidad_Cuencas_2020_lite.json
[ÉXITO] Archivo serializado: /home/sebs/Documentos/HackODS/linuxitOS-HackODS2026/datos/Disponibilidad_Cuencas_2020_lite.json
[INFO] Peso heurístico en disco: 1.00 MB

[INFO] Cargando topología: Estado.json
[INFO] Aplicando Douglas-Peucker: reduccion poligonal para Estado.json
[INFO] Vaciando vector espacial optimizado a io stream: Estado_lite.json
[ÉXITO] Archivo serializado: /home/sebs/Documentos/HackODS/linuxitOS-HackODS2026/datos/Estado_lite.json
[INFO] Peso heurístico en disco: 0.22 MB


[ÉXITO] Pipeline ETL concluido satisfactoriamente.
[INFO] El dataset analítico 'final_merged_data.csv' ha sido generado exitosamente.
[INFO] Para compilar el dashboard Quarto, ejecute el siguiente comando:
       quarto render dashboard/index.qmd


## Conclusión del Pipeline

Al ejecutar el orquestador `main.py`, se generan los datasets limpios y el archivo geoespacial optimizado en la carpeta `datos/`. Estos archivos son la columna vertebral y fuente de la verdad para nuestro tablero de Quarto.

El archivo analítico final consolidado es: `datos/final_merged_data.csv`.
